In [1]:
from pyspark.sql import SparkSession 
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, sum, mean , min , max , count , lit, lower, upper, initcap
from datetime import date , datetime

from functools import reduce                             

In [2]:
spark = SparkSession.builder.getOrCreate()

In [3]:
martBranches = spark.read.csv("mart_branches.csv" , header = 'True')
martCust = spark.read.csv("mart_customers.csv", header = 'True')
martProd = spark.read.csv("mart_products.csv", header = 'True')
martProm = spark.read.csv("mart_promotions.csv", header = 'True')
martSales = spark.read.csv("mart_sales_master.csv", header = 'True')

In [4]:
'''
now doing : martBranches
'''
martBranches.show() # print dataframe

+-----------+--------------+
|branch_code|   branch_name|
+-----------+--------------+
|         PJ| Petaling Jaya|
|         SB|   Subang Jaya|
|         KL|  Kuala Lumpur|
|         CH|        Cheras|
|         IP|          Ipoh|
|         PG|        Penang|
|         JB|   Johor Bahru|
|         PJ|Petaling Jayaa|
|         KL|  Kuala Lumper|
+-----------+--------------+



In [5]:
martBranches.printSchema()  
##dual string is fine

root
 |-- branch_code: string (nullable = true)
 |-- branch_name: string (nullable = true)



In [6]:
martBranches = martBranches.dropDuplicates(["branch_code"])
martBranches.show()
#no more duplicates

+-----------+-------------+
|branch_code|  branch_name|
+-----------+-------------+
|         CH|       Cheras|
|         IP|         Ipoh|
|         JB|  Johor Bahru|
|         KL| Kuala Lumpur|
|         PG|       Penang|
|         PJ|Petaling Jaya|
|         SB|  Subang Jaya|
+-----------+-------------+



In [7]:
##MART CUST NOW
martCust.show(20)
##null values
total_rows_cust= martCust.count()
#for later usage

+-----------+------+----+-----------+------------+
|customer_id|gender| age|member_tier|member_since|
+-----------+------+----+-----------+------------+
|      C1000|     F|NULL|       Gold|  2021-12-15|
|      C1001|     M|  57|     Silver|  2021-09-28|
|      C1002|  NULL|  41|       Gold|  2022-06-25|
|      C1003|     F|  30|       Gold|  2022-12-26|
|      C1004|  NULL|  43|       Gold|  2024-06-18|
|      C1005|     M|  52|     Silver|  2023-10-24|
|      C1006|     F|  68|     Silver|  2022-10-26|
|      C1007|     F|  33|     Silver|  2021-12-21|
|      C1008|     F|  45|     Silver|  2023-11-05|
|      C1009|     F|  36|     Silver|  2021-04-09|
|      C1010|     F|  59|     Silver|  2021-06-07|
|      C1011|     M|  19|       Gold|  2022-12-09|
|      C1012|     M|  46|   Platinum|  2024-03-15|
|      C1013|     F|  30|       NULL|  2023-12-11|
|      C1014|     M|  68|     Silver|  2023-09-12|
|      C1015|     F|  62|       NULL|  2022-04-04|
|      C1016|  NULL|  42|     S

In [8]:
martCust.printSchema()
martCust.describe().show()
#change age to num
#change member_since to date

root
 |-- customer_id: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: string (nullable = true)
 |-- member_tier: string (nullable = true)
 |-- member_since: string (nullable = true)

+-------+-----------+------+------------------+-----------+------------+
|summary|customer_id|gender|               age|member_tier|member_since|
+-------+-----------+------+------------------+-----------+------------+
|  count|       1800|  1683|              1739|       1509|        1800|
|   mean|       NULL|  NULL| 45.64059804485336|       NULL|        NULL|
| stddev|       NULL|  NULL|20.751258142261786|       NULL|        NULL|
|    min|      C1000|     F|               150|       Gold|  2021-01-01|
|    max|      C2799|     M|                69|     Silver|  2025-01-01|
+-------+-----------+------+------------------+-----------+------------+



In [9]:
martCust = martCust.withColumn("age", col("age").cast("integer"))
martCust = martCust.withColumn("member_since", col("member_since").cast("date"))
martCust.printSchema()
#types change , something something

root
 |-- customer_id: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- member_tier: string (nullable = true)
 |-- member_since: date (nullable = true)



In [10]:
distinct = martCust.distinct().count()
print(f"Duplicate rows: {total_rows_cust - distinct}")

Duplicate rows: 0


In [11]:

#checking skew + quartiles
martCust.select(
    F.skewness("age").alias("age_skew")
).show()
martCust.select(
    F.percentile_approx("age", [0, 0.25, 0.5, 0.75, 1]).alias("quartiles")
).show(truncate=False)

#150 :) 

#used to be below checking null , now is above , if shit dont make sense @me

+------------------+
|          age_skew|
+------------------+
|2.1757073389747315|
+------------------+

+---------------------+
|quartiles            |
+---------------------+
|[18, 30, 45, 57, 150]|
+---------------------+



In [12]:

print(martCust.filter("age = 150").count())
#why are there 32 ppl 150 age 
martCust = martCust.withColumn(
    "age",
    F.when(
        (F.col("age") >= 150),
        None
    ).otherwise(F.col("age"))
)

print(martCust.filter("age = 150").count())

32
0


In [13]:
martCust.select(
    F.skewness("age").alias("age_skew")
).show()
martCust.select(
    F.percentile_approx("age", [0, 0.25, 0.5, 0.75, 1]).alias("quartiles")
).show(truncate=False)
#surely its fixed now, seems good enough

+--------------------+
|            age_skew|
+--------------------+
|-0.04500051970636024|
+--------------------+

+--------------------+
|quartiles           |
+--------------------+
|[18, 30, 45, 57, 69]|
+--------------------+



In [14]:
martCust.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in martCust.columns
]).show()

#count no. null values

+-----------+------+---+-----------+------------+
|customer_id|gender|age|member_tier|member_since|
+-----------+------+---+-----------+------------+
|          0|   117| 93|        291|           0|
+-----------+------+---+-----------+------------+



In [15]:
martCust.select([
    (
        F.count(F.when(F.col(c).isNull(), c))
        / total_rows_cust * 100
    ).alias(c)
    for c in martCust.columns
]).show()

#null value percentage
#so the question is : what does null values in member_tier mean
#either no data , or  not a tier , i dont think any other attribute correlates to member tier 
#however sales_master does refrence customer id 
#so plan is to not drop any rows
#fill gender with Mode , Age with Median , replace null in membertier with Unmembered
#other option is to drop member_tier directly , does only show up in martCust

+-----------+------+-----------------+------------------+------------+
|customer_id|gender|              age|       member_tier|member_since|
+-----------+------+-----------------+------------------+------------+
|        0.0|   6.5|5.166666666666667|16.166666666666664|         0.0|
+-----------+------+-----------------+------------------+------------+



In [16]:
#fill gender with Mode
mode_val = martCust.groupBy("gender").count().orderBy("gender", ascending=False).first()[0]
martCust = martCust.fillna({"gender": mode_val})

In [17]:
#fill age with mean ,ttfs
mean_val = martCust.select(mean("age")).first()[0]
martCust = martCust.fillna({"age": mean_val})

In [18]:
#I will assume Null means Basic Member
#But in the case of needing to drop

#martCust = masrtCust.dropna(subset=["member_tier"])


In [19]:
martCust.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in martCust.columns
]).show()
# only member tier remains
#fuck i did not forgot duplicates

+-----------+------+---+-----------+------------+
|customer_id|gender|age|member_tier|member_since|
+-----------+------+---+-----------+------------+
|          0|     0|  0|        291|           0|
+-----------+------+---+-----------+------------+



In [20]:
#final checks to be safe
martCust.groupBy("gender").count().show()

martCust.filter(
    F.col("member_since") > F.lit("2026-06-01")
).show()

martCust.groupBy("customer_id") \
        .count() \
        .filter("count > 1") \
        .show()

#no inconsitencies , go next 

+------+-----+
|gender|count|
+------+-----+
|     F|  843|
|     M|  957|
+------+-----+

+-----------+------+---+-----------+------------+
|customer_id|gender|age|member_tier|member_since|
+-----------+------+---+-----------+------------+
+-----------+------+---+-----------+------------+

+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+



In [21]:
## MART PROD
martProd.show()

total_rows_prod = martProd.count()
print (total_rows_prod)
## so , basic data checking
#no null , unqiue product_ids , categoris are fine , price is fine 
#also dont see any dupes
#just datatypes imo

+----------+--------------------+--------+---------+----------+
|product_id|        product_name|category|std_price|cost_price|
+----------+--------------------+--------+---------+----------+
|    BEV001|      Coca Cola 1.5L|Beverage|      4.5|       3.8|
|    BEV002| Mineral Water 500ml|Beverage|      1.8|       0.6|
|    BEV003|     Orange Juice 1L|Beverage|      7.9|       5.9|
|    SNK001|    Potato Chips BBQ|   Snack|      6.9|       3.0|
|    SNK002|Chocolate Cookies...|   Snack|      8.2|       4.6|
|    HOM001|       Detergent 2kg|    Home|     18.9|      15.2|
|    HOM002|Dishwashing Liqui...|    Home|      9.5|       6.1|
|    FRS001|      Apple Fuji 1kg|   Fresh|      8.5|       7.0|
|    FRS002|          Banana 1kg|   Fresh|      5.5|       4.8|
|    PER001|       Shampoo 650ml|Personal|     15.9|      10.4|
|    PER002|        Body Wash 1L|Personal|     14.2|       9.7|
|    FRZ001|  Chicken Nugget 1kg|  Frozen|     14.9|      11.8|
|    SNK001|Potato Chips Barb...|   Snac

In [22]:
distinct = martProd.distinct().count()
print(f"Duplicate rows: {total_rows_prod - distinct}")
#be safe?

Duplicate rows: 0


In [23]:
martProd.printSchema()  

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- std_price: string (nullable = true)
 |-- cost_price: string (nullable = true)



In [24]:
martProd = martProd.withColumn("std_price", col("std_price").cast("float"))
martProd = martProd.withColumn("cost_price", col("cost_price").cast("float"))

In [25]:
martProd.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- std_price: float (nullable = true)
 |-- cost_price: float (nullable = true)



In [26]:
## MART PROMOTION
martProm.show()

#realisitcally , do nothing
#only whitespace inconsitencies?

+----------+--------------------+
|promo_code|         description|
+----------+--------------------+
|      NONE|            No Promo|
|   MEMBER8|           Member 8%|
|     CNY10|             CNY 10%|
|YEAR_END15|        Year End 15%|
|    LOSS30|     Loss Leader 30%|
|  BADPROMO|Corrupted Promo Code|
+----------+--------------------+



In [27]:
## MART SALES

martSales.show()

total_rows_sales = martSales.count()
print(total_rows_sales)
#das alot of rows 71654

##to do ensure datypes are valid

+----------+-------------------+------------+----------+--------------------+---+----------+----------+-----------+
|invoice_id|            txn_raw|      branch|product_id|        product_name|qty|unit_price|promo_code|customer_id|
+----------+-------------------+------------+----------+--------------------+---+----------+----------+-----------+
| INV100000|   19/11/2025 21:34| PetalingJya|    SNK002|ChocolateCookies200g|  3|      7.38|     CNY10|      C1220|
| INV100001|   08-03-2025 10:32| johor bahru|    BEV001|           Coke 1.5L|  2|       4.5|      NONE|      C1680|
| INV100002|2025-08-13 18:19:00|KUALA LUMPUR|    BEV001|        CocaCola1.5L|  1|      4.05|     CNY10|      C2148|
| INV100003|2025-11-02 16:03:00|KUALA LUMPUR|    FRZ001|  Chicken Nugget 1kg|  2|     13.71|   MEMBER8|      C1893|
| INV100004|   06-26-2025 16:41|          JB|    BEV002| Mineral Water 500ml|  2|       1.8|      NONE|      C1646|
| INV100005|   02/08/2025 15:30|          PG|    FRS001|      Apple Fuji

In [28]:
martSales.printSchema()
# so 
# invoice - fine
# txn - date time fix formatting
# branch - string fix formatting
# prod_id -fine
# name - fine , check ending seq
# qty - CHANGE int
# price CHANGE float
# code fine
# custID - fine

root
 |-- invoice_id: string (nullable = true)
 |-- txn_raw: string (nullable = true)
 |-- branch: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- qty: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- promo_code: string (nullable = true)
 |-- customer_id: string (nullable = true)



In [29]:
##checking for null first
#data type shenneingans will check other later
martSales.select(
    F.count(
        F.when(F.col("txn_raw").isNull(), 1)
    ).alias("txn_raw_nulls")
).show()

+-------------+
|txn_raw_nulls|
+-------------+
|          559|
+-------------+



In [30]:
# txn_raw

#19/11/2025 21:34
#dd/MM/yyyy HH:mm

#2025-11-02 16:03:00
#yyyy-MM-dd HH:mm:ss

#Apr 14 2025 18:34
#MMM dd yyyy HH:mm

#06-26-2025 16:41
#MM-dd-yyyy HH:mm

#2025-11-19 21:34:00
#yyyy-MM-dd HH:mm:ss

#31/02/2025 10:00
#ddMMyyyy HH:mm


martSales = martSales.withColumn(
    "txn_cooked",
    F.coalesce(
        F.to_timestamp("txn_raw", "dd/MM/yyyy HH:mm"),
        F.to_timestamp("txn_raw", "yyyy-MM-dd HH:mm:ss"),
        F.to_timestamp("txn_raw", "MMM dd yyyy HH:mm"),
        F.to_timestamp("txn_raw", "MM-dd-yyyy HH:mm")
    
    )
)
##martSales = martSales.withColumn("txn_raw", col("txn_raw").cast("timestamp"))

#just checking if txn_has values that nulled

##validating dates all are 
print(
    martSales.filter(
        F.col("txn_cooked").isNull()
    ).count()
)


martSales.filter(
    F.col("txn_cooked").isNull()
).groupBy("txn_raw").count().show(100, False)
'''
|31/02/2025 10:00|61   |  - invalide day
|NULL            |559  |  - null
|notadate        |56   |  - not a date
|2025-99-01      |50   |  - invalide month
|32/13/2025      |53   |  - invalide date and month
|yesterday       |71   |  - when was yesterday
|???             |1    |  - ???
|2025/88/12      |1    |  - invalude month date in a diffrent format
'''
## so turn all to null

martSales.filter(
    F.col("txn_cooked").isNull()
).groupBy("txn_cooked").count().show(100, False)

martSales = (
    martSales
    .drop("txn_raw")
    .withColumn("transaction DateTime" , F.col("txn_cooked"))
    .drop("txn_cooked")
)

martSales.show()

852
+----------------+-----+
|txn_raw         |count|
+----------------+-----+
|31/02/2025 10:00|61   |
|NULL            |559  |
|notadate        |56   |
|2025-99-01      |50   |
|32/13/2025      |53   |
|yesterday       |71   |
|???             |1    |
|2025/88/12      |1    |
+----------------+-----+

+----------+-----+
|txn_cooked|count|
+----------+-----+
|NULL      |852  |
+----------+-----+

+----------+------------+----------+--------------------+---+----------+----------+-----------+--------------------+
|invoice_id|      branch|product_id|        product_name|qty|unit_price|promo_code|customer_id|transaction DateTime|
+----------+------------+----------+--------------------+---+----------+----------+-----------+--------------------+
| INV100000| PetalingJya|    SNK002|ChocolateCookies200g|  3|      7.38|     CNY10|      C1220| 2025-11-19 21:34:00|
| INV100001| johor bahru|    BEV001|           Coke 1.5L|  2|       4.5|      NONE|      C1680| 2025-08-03 10:32:00|
| INV100002|KU

In [31]:
##Branch Time
#is string so is fine

#check nulls
martSales.select(
    F.count(
        F.when(F.col("branch").isNull(), 1)
    ).alias("branch")
).show()

martSales.groupBy("branch").count().show(100, False)
# building a long ass logic sequence
#dont see branch ID
martSales = martSales.replace(
    {
        "Petaling Jaya": "PJ",
        "Petaling Jayaa": "PJ",
        "PetalingJaya": "PJ",
        "PetalingJya": "PJ",
        "PETALING JAYA": "PJ",
        "petaling jaya": "PJ",
        "P.J.": "PJ",
        "Subang": "SB",
        "SubangJaya": "SB",
        "Subang Jaya": "SB",
        "subang jaya": "SB",
        "SUBANG JAYA": "SB",
        "Subng Jaya": "SB",
        "Kuala Lumpur": "KL",
        "kuala lumpur": "KL",
        "KUALA LUMPUR": "KL",
        "KualaLumpur": "KL",
        "Kuala Lumper": "KL",
        "K.L.": "KL",
        "Johor Bahru": "JB",
        "JOHOR BAHRU": "JB",
        "johor bahru": "JB",
        "JohorBahru": "JB",
        "Penang": "PG",
        "penang": "PG",
        "PENANG": "PG",
        "Ipoh": "IP",
        "IPOH": "IP",
        "ipoh": "IP",
        "Cheras": "CH",
        "CHERAS": "CH",
        "cheras": "CH",
        "ERROR": None,
        "UNKNOWN": None
    },
    subset=["branch"]
)

## can definitely be optimised
## Loop through the mart_bracnhes name
## convert thorugh transofmration , match withname ,then give value 2L
#

martSales.groupBy("branch").count().show(100, False)

+------+
|branch|
+------+
|   801|
+------+

+--------------+-----+
|branch        |count|
+--------------+-----+
|IPOH          |1963 |
|SB            |1398 |
|johor bahru   |1984 |
|Petaling Jaya |1450 |
|Petaling Jayaa|1497 |
|Subang        |1353 |
|Kuala Lumpur  |1256 |
|NULL          |801  |
|penang        |1934 |
|PG            |1975 |
|SubangJaya    |1382 |
|KL            |1193 |
|kuala lumpur  |1172 |
|SUBANG JAYA   |1356 |
|JohorBahru    |1938 |
|KualaLumpur   |2436 |
|KUALA LUMPUR  |1242 |
|K.L.          |1225 |
|subang jaya   |1411 |
|ipoh          |1955 |
|CH            |1965 |
|JB            |1925 |
|Penang        |3888 |
|PetalingJya   |1541 |
|PENANG        |1935 |
|PETALING JAYA |1510 |
|PetalingJaya  |1560 |
|Subang Jaya   |1331 |
|JOHOR BAHRU   |2013 |
|P.J.          |1504 |
|Subng Jaya    |1411 |
|PJ            |1548 |
|IP            |1963 |
|Johor Bahru   |2003 |
|CHERAS        |2014 |
|Cheras        |3881 |
|Ipoh          |3980 |
|Kuala Lumper  |1229 |
|cheras    

In [32]:
## qty 
martSales.groupBy("qty") \
        .count() \
        .show(10000)

+----+-----+
| qty|count|
+----+-----+
|  -4|  149|
|  51|    1|
|  -1| 2135|
| two|   49|
| 101|    1|
|  -6|    2|
|  69|    2|
|  29|    2|
|  42|    1|
| -39|    2|
|   3|10544|
| 113|    1|
|  34|    1|
|  28|    2|
|  22|    2|
|  35|    1|
| -22|    3|
|  71|    1|
|  99|    2|
| 107|    1|
| -29|    3|
|  -8|    1|
|NULL|  744|
|   5| 1655|
| 100|    1|
|-999|   40|
|  27|    2|
|  75|    1|
|  46|    3|
| -23|    3|
| -10|    5|
|   6|  845|
| 118|    2|
| -25|    1|
| -41|    3|
|  68|    1|
|  90|    1|
| 104|    1|
|  41|    1|
| 102|    1|
| -42|    6|
| -19|    2|
| 111|    1|
| -12|    1|
|  95|    1|
| -32|    2|
|  81|    1|
| -16|    1|
| 114|    1|
| -44|    4|
| -43|    1|
|  48|    2|
| -35|    1|
| 1.5|   52|
|  67|    1|
|  84|    2|
|9999|   44|
|  79|    2|
|  24|    1|
|  88|    2|
|   1|28667|
| -11|    4|
| -18|    5|
|  36|    1|
| abc|   49|
|  37|    2|
| -20|    1|
|  49|    1|
| -33|    4|
|  -3|  154|
| -28|    4|
| -24|    2|
|  65|    3|
|   4| 7742|

In [33]:
# qty - CHANGE int
martSales = martSales.withColumn("qty", col("qty").cast("integer"))

In [34]:
# price
martSales.groupBy("unit_price") \
        .count() \
        .show(10000)
##RM presumabably just remove it
# so remove RM and 
# free = 0

+----------+-----+
|unit_price|count|
+----------+-----+
|       8.5| 2907|
|       8.2| 2814|
|      7.65|  730|
|      14.2| 2891|
|      1.53|  886|
|    RM9.94|   13|
|    RM4.05|   16|
|    RM5.95|    5|
|    RM15.9|   44|
|    RM14.2|   41|
|    999.99|   69|
|    RM1.66|   23|
|   RM13.06|   14|
|   RM17.01|   12|
|    RM6.21|   20|
|      9.94|  495|
|    RM3.15|    8|
|   RM12.07|   11|
|    RM1.26|    7|
|    RM8.74|   10|
|    RM8.07|   11|
|       4.5| 2873|
|      5.06|  860|
|    RM3.82|   17|
|     11.13|  456|
|    RM5.53|    6|
|    RM6.65|    4|
|      5.95|  495|
|     13.71|  901|
|     12.66|  904|
|      NULL|  901|
|   RM14.31|    9|
|      3.85|  463|
|    RM7.65|   13|
|     RM8.2|   59|
|      1.66|  905|
|      8.07|  828|
|    RM1.53|   21|
|      3.15|  464|
|    RM14.9|   58|
|      6.35|  870|
|    RM8.55|    8|
|       1.8| 2792|
|      4.14|  839|
|     16.06|  893|
|     13.52|  889|
|      4.95|  727|
|     RM7.9|   48|
|      8.55|  728|
|       9.5|

In [35]:
# price CHANGE float

martSales = martSales.withColumn("unit_price",
    when(col("unit_price").startswith("RM"), col("unit_price").substr(3, 100).cast("float"))
    .when(col("unit_price") == "free", lit(0).cast("float"))
    .otherwise(col("unit_price").cast("float"))
)

martSales.groupBy("unit_price") \
        .count() \
        .show(10000)


+----------+-----+
|unit_price|count|
+----------+-----+
|       6.9| 2961|
|      5.06|  878|
|      4.95|  732|
|       5.5| 2887|
|      4.14|  847|
|       8.5| 2965|
|      14.2| 2932|
|     13.41|  659|
|     17.39|  939|
|      4.67|  836|
|      7.38|  734|
|       9.5| 2916|
|      NULL|  902|
|      5.53|  440|
|     13.23|  448|
|       1.8| 2837|
|      14.9| 2840|
|      1.53|  907|
|     16.06|  896|
|     13.06|  900|
|      1.26|  434|
|       8.2| 2873|
|      7.54|  864|
|     12.07|  922|
|    999.99|   69|
|     12.78|  673|
|      8.74|  862|
|     10.43|  468|
|      1.62|  700|
|     14.31|  728|
|      15.9| 3004|
|      3.85|  472|
|      5.95|  500|
|      5.87|  944|
|      3.15|  472|
|     12.66|  923|
|      5.74|  437|
|     11.13|  463|
|      7.65|  743|
|      18.9| 2882|
|      3.82|  842|
|     13.52|  898|
|      6.35|  894|
|      7.82|  905|
|      9.94|  508|
|      7.22|  887|
|     14.63|  856|
|       4.5| 2919|
|     13.71|  914|
|      8.55|

In [36]:
martSales.printSchema()

root
 |-- invoice_id: string (nullable = true)
 |-- branch: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- unit_price: float (nullable = true)
 |-- promo_code: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- transaction DateTime: timestamp (nullable = true)



In [37]:
##check dupe rows 
distinct = martSales.distinct().count()
print(f"Duplicate rows: {total_rows_sales - distinct}")

martSales = martSales.distinct()

print(f"After : {martSales.count() - distinct}")


Duplicate rows: 1185
After : 0


In [38]:
#Key Validation

martSales.groupBy("invoice_id") \
        .count() \
        .filter("count > 1") \
        .show()

martSales.filter("invoice_id = 'INV112753'").show()
martSales.filter("invoice_id = 'INV110049'").show()



#wtf do maths?
#OJ 1l = 7.9
#so math is total * discount / sum 
#so 7.9 * 5 / *0.98 / 5 = 7.742 ???
#so 7.9 * 10 / *0.98 / 10  = 7.742 ???
# cs - std ? 
# (10(7.9) - 10(5.9))*98%
#???

+----------+-----+
|invoice_id|count|
+----------+-----+
| INV112753|    2|
| INV157363|    2|
| INV125692|    2|
| INV128725|    2|
| INV160896|    2|
| INV110049|    2|
| INV105027|    2|
| INV139037|    2|
| INV135680|    2|
| INV126315|    2|
| INV117085|    2|
| INV152064|    2|
| INV153628|    2|
| INV120311|    2|
| INV163371|    2|
| INV166713|    2|
| INV160838|    2|
| INV161052|    2|
| INV126385|    2|
| INV107467|    2|
+----------+-----+
only showing top 20 rows

+----------+------+----------+---------------+---+----------+----------+-----------+--------------------+
|invoice_id|branch|product_id|   product_name|qty|unit_price|promo_code|customer_id|transaction DateTime|
+----------+------+----------+---------------+---+----------+----------+-----------+--------------------+
| INV112753|    CH|    BEV003|ORANGE JUICE 1L|  5|      7.27|   MEMBER8|      C2147| 2025-11-08 16:33:00|
| INV112753|    CH|    BEV003|ORANGE JUICE 1L|  7|      10.9|   MEMBER8|      C2147| 2025-11-0

In [39]:
martSales.groupBy("branch") \
        .count() \
        .show()

+------+-----+
|branch|count|
+------+-----+
|    SB| 9482|
|  NULL|  803|
|    PG| 9580|
|    KL| 9599|
|    CH| 9674|
|    JB| 9693|
|    PJ|11945|
|    IP| 9693|
+------+-----+



In [40]:
martSales.groupBy("product_id") \
        .count() \
        .show()
'''	product_id
1	BEV001
2	BEV002
3	BEV003
4	SNK001
5	SNK002
6	HOM001
7	HOM002
8	FRS001
9	FRS002
10	PER001
11	PER002
12	FRZ001
13	SNK001'''

+----------+-----+
|product_id|count|
+----------+-----+
|    PER001| 5874|
|    HOM001| 5815|
|    SNK002| 5692|
|      NULL|  710|
|    FRS001| 5917|
|    FRZ001| 5736|
|    FRS002| 5755|
|    BEV001| 5702|
|    XXX001|   56|
|   MISSING|    1|
|    BEV999|   58|
|    PRD999|   46|
|       abc|    1|
|    HOM002| 5769|
|    PER002| 5855|
|    SNK001| 5913|
|    BEV002| 5726|
|    BEV003| 5843|
+----------+-----+



'\tproduct_id\n1\tBEV001\n2\tBEV002\n3\tBEV003\n4\tSNK001\n5\tSNK002\n6\tHOM001\n7\tHOM002\n8\tFRS001\n9\tFRS002\n10\tPER001\n11\tPER002\n12\tFRZ001\n13\tSNK001'

In [55]:
martSales.groupBy("product_name") \
        .count() \
        .show(100)
'''
merge on product id
'''
#martProd("product_id")
#martProd("product_name")
products = martProd.withColumnRenamed("product_name", "product_name_ref") \
                   .withColumnRenamed("product_id", "product_id_ref")

martSales = martSales.join(products, martSales.product_id == martProd.product_id_ref, how="left") \
       .withColumn("product_name", 
           when(col("product_name_ref").isNotNull(), col("product_name_ref"))
           .otherwise(col("product_name"))) \
       .drop("product_id_ref", "product_name_ref")


martSales.groupBy("product_name") \
        .count() \
        .show(100)

+--------------------+-----+
|        product_name|count|
+--------------------+-----+
|        Body Wash 1l|   59|
|          Bodywash1l|   17|
|Chocolatecookies200g|   16|
|      Coca Cola 1.5l|   27|
|           Bbq Chips|   13|
|        Body Wash 1L| 5855|
|     Orange Juice 1L| 5843|
|        Applefuji1kg|   22|
|                NULL|    7|
|        Detergent2kg|   21|
|   Mineralwater500ml|   18|
|       Shampoo 650ml| 5926|
|      Potatochip Bbq|   11|
|     Orange Juice 1l|   50|
|                 N/a|    4|
|             Unknown|    7|
|      Potatochipsbbq|   13|
|    Potato Chips Bbq|   31|
| Coca Cola 1.5 Liter|   11|
| Mineral Water 500ml| 5789|
|       Orangejuice1l|   17|
|    Chickennugget1kg|   16|
|      Apple Fuji 1kg| 5951|
|        Null Product|    1|
|           Banana1kg|   12|
|Chocolate Cookies...| 5745|
|Dishwashing Liqui...| 5833|
|           Coke 1.5l|   11|
|Dishwashingliquid...|   14|
|                 ???|    1|
|      Coca Cola 1.5L| 5702|
|    Potato Ch

AttributeError: 'DataFrame' object has no attribute 'product_id_ref'

In [42]:
martSales.groupBy("promo_code") \
        .count() \
        .show()

''' expected values
1	NONE
2	MEMBER8
3	CNY10
4	YEAR_END15
5	LOSS30
6	BADPROMO
'''

+----------+-----+
|promo_code|count|
+----------+-----+
|YEAR_END15|10719|
|      NULL|    1|
|    LOSS30| 5574|
|  BADPROMO|    1|
|   MEMBER8|10623|
|     CNY10| 8547|
|      NONE|35004|
+----------+-----+



' expected values\n1\tNONE\n2\tMEMBER8\n3\tCNY10\n4\tYEAR_END15\n5\tLOSS30\n6\tBADPROMO\n'

In [43]:
martSales.groupBy("customer_id") \
        .count() \
        .show()

+-----------+-----+
|customer_id|count|
+-----------+-----+
|      C2100|   46|
|      C2367|   32|
|      C2207|   50|
|      C1602|   30|
|      C2320|   36|
|      C1524|   26|
|      C1804|   39|
|      C2430|   41|
|      C1100|   28|
|      C2777|   34|
|      C1875|   31|
|      C1571|   30|
|      C1842|   31|
|      C1829|   33|
|      C2490|   42|
|      C1628|   35|
|      C1305|   54|
|      C2243|   44|
|      C1774|   33|
|      C1194|   43|
+-----------+-----+
only showing top 20 rows

